# Beginner: Run Easy SeqTrainer BenchLab Models From JSON

This notebook shows the easiest path for running the lightweight local models from a saved `run_config.json`.

It runs:

- Linear Regression
- Random Forest
- Gradient Boosting

The important idea: **the model settings come from JSON**, not from hidden notebook variables.

## Step 0: What This Notebook Uses

This notebook uses a tiny example dataset and config already included in the repo:

- Dataset: `examples/reproducibility/easy_promoters.csv`
- Config: `examples/reproducibility/easy_run_config.json`

Later, you can replace the config path with a `run_config.json` exported by BenchLab.

In [1]:
from pathlib import Path
import sys

# Make imports work whether this notebook is opened from the repo root or from notebooks/.
repo_root = Path.cwd()
if not (repo_root / "app").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)

Repo root: c:\Users\Sgoff\MYfile\Desktop\PYThh\seqtrainer-benchlab


## Step 1: Load The JSON Config

The config stores the dataset path, checksum, preprocessing choices, model list, split seed, and training settings.

In [2]:
from app.reproducibility.config import ReproducibleRunConfig

config_path = repo_root / "examples" / "reproducibility" / "easy_run_config.json"
config = ReproducibleRunConfig.load(config_path)
dataset_path_from_config = Path(config.dataset.path)
if not dataset_path_from_config.is_absolute():
    config.dataset.path = str((repo_root / dataset_path_from_config).resolve())

print("Schema version:", config.schema_version)
print("Dataset:", config.dataset.path)
print("Sequence column:", config.dataset.sequence_column)
print("Target column:", config.dataset.target_column)
print("Models:", [model.model_name for model in config.models])
print("Random seed:", config.split.random_seed)

Schema version: 1.0.0
Dataset: C:\Users\Sgoff\MYfile\Desktop\PYThh\seqtrainer-benchlab\examples\reproducibility\easy_promoters.csv
Sequence column: sequence
Target column: label
Models: ['linear_regression', 'random_forest', 'gradient_boosting']
Random seed: 42


## Step 2: Check The Dataset

BenchLab stores a SHA256 checksum in the JSON. If the local dataset exists, we can confirm it matches the saved config.

In [3]:
from app.reproducibility.run_from_config import verify_dataset_checksum

print("Checksum verified:", verify_dataset_checksum(config))

Checksum verified: True


## Step 3: Preview The Dataset

This is just to help you see what the example data looks like.

In [4]:
import pandas as pd

dataset_path = Path(config.dataset.path)
df = pd.read_csv(dataset_path)
df.head()
df.describe()

,label
count,12.000000
mean,0.500000
std,0.522233
min,0.000000
25%,0.000000
50%,0.500000
75%,1.000000
max,1.000000


## Step 4: Dry Run First

A dry run validates the config and writes reproducibility files, but it does not train models. This is a safe first check.

In [5]:
from app.reproducibility.run_from_config import replay_from_config

dry_output = repo_root / "storage" / "notebook_beginner_dry_run"
dry_summary = replay_from_config(config_path, output_dir=dry_output, dry_run=True)
dry_summary

{'dry_run': True,
 'output_dir': 'c:\\Users\\Sgoff\\MYfile\\Desktop\\PYThh\\seqtrainer-benchlab\\storage\\notebook_beginner_dry_run',
 'schema_version': '1.0.0',
 'dataset_checksum_verified': True,
 'models': ['linear_regression', 'random_forest', 'gradient_boosting'],
 'random_seed': 42,
 'preprocessing': {'use_gc': True,
  'use_kmers': True,
  'normalize_kmers': True,
  'use_one_hot': False,
  'kmer_size': 6,
  'sequence_length': 150,
  'additional_options': {}}}

## Step 5: Train The Easy Models

Now we run the supported local models from the same JSON config.

This creates metrics, predictions, model files, and reproducibility artifacts.

In [6]:
run_output = repo_root / "storage" / "notebook_beginner_easy_run"
run_summary = replay_from_config(config_path, output_dir=run_output, dry_run=False)
run_summary

{'dry_run': False,
 'output_dir': 'c:\\Users\\Sgoff\\MYfile\\Desktop\\PYThh\\seqtrainer-benchlab\\storage\\notebook_beginner_easy_run',
 'schema_version': '1.0.0',
 'dataset_checksum_verified': True,
 'models': ['linear_regression', 'random_forest', 'gradient_boosting'],
 'random_seed': 42,
 'preprocessing': {'use_gc': True,
  'use_kmers': True,
  'normalize_kmers': True,
  'use_one_hot': False,
  'kmer_size': 6,
  'sequence_length': 150,
  'additional_options': {}},
 'runnable_models': ['linear_regression',
  'random_forest',
  'gradient_boosting'],
 'skipped_models': [],
 'metrics_path': 'c:\\Users\\Sgoff\\MYfile\\Desktop\\PYThh\\seqtrainer-benchlab\\storage\\notebook_beginner_easy_run\\metrics.json',
 'predictions_path': 'c:\\Users\\Sgoff\\MYfile\\Desktop\\PYThh\\seqtrainer-benchlab\\storage\\notebook_beginner_easy_run\\predictions.csv'}

In [7]:
print("Checksum verified:", verify_dataset_checksum(config))

Checksum verified: True


## Step 6: Read The Metrics

The output metrics are saved as JSON. For binary labels, BenchLab also reports thresholded metrics such as accuracy, precision, recall, F1, and MCC.

In [8]:
import json

metrics_path = Path(run_summary["metrics_path"])
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
metrics

{'gradient_boosting': {'accuracy': 0.6666666666666666,
  'classification_threshold': 0.5,
  'f1': 0.0,
  'false_negative': 1,
  'false_positive': 0,
  'mae': 0.3333382521109051,
  'mcc': 0.0,
  'mse': 0.3333234959959392,
  'precision': 0.0,
  'r2': -0.49995573198172605,
  'recall': 0.0,
  'rmse': 0.5773417497426798,
  'task_type': 'binary_label_regression_baseline',
  'test_rows': 3,
  'threshold_strategy': 'user_or_literature',
  'train_rows': 9,
  'true_negative': 2,
  'true_positive': 0},
 'linear_regression': {'accuracy': 1.0,
  'classification_threshold': 0.5,
  'f1': 1.0,
  'false_negative': 0,
  'false_positive': 0,
  'mae': 0.2907348242811511,
  'mcc': 1.0,
  'mse': 0.09352958588941442,
  'precision': 1.0,
  'r2': 0.5791168634976351,
  'recall': 1.0,
  'rmse': 0.3058260713042863,
  'task_type': 'binary_label_regression_baseline',
  'test_rows': 3,
  'threshold_strategy': 'user_or_literature',
  'train_rows': 9,
  'true_negative': 2,
  'true_positive': 1},
 'random_forest': {'ac

## Step 7: Read The Predictions

Predictions are saved as CSV so they can be compared later with CNN, DNABERT2, or iPro-MP outputs.

In [9]:
predictions_path = Path(run_summary["predictions_path"])
predictions = pd.read_csv(predictions_path)
predictions.head()

,model,actual,prediction,predicted_label
0,linear_regression,0,0.223642,0
1,linear_regression,1,0.575080,1
2,linear_regression,0,0.223642,0
3,random_forest,0,0.190000,0
4,random_forest,1,0.650000,1


## Step 8: What Files Were Created?

These files help reproduce or audit the run later.

In [10]:
for path in sorted(run_output.iterdir()):
    print(path.name)

Dockerfile.repro
environment.yml
gradient_boosting.joblib
linear_regression.joblib
metrics.json
predictions.csv
random_forest.joblib
replay_summary.json
requirements.lock.txt
run_config.json


## Step 9: Use Your Own BenchLab Export

After running BenchLab in the web app, download/export a run bundle and find its `run_config.json`.

Then replace this line:

```python
config_path = repo_root / "examples" / "reproducibility" / "easy_run_config.json"
```

with your own config path, for example:

```python
config_path = Path("C:/path/to/my/run_config.json")
```

Important: if you want to train from the JSON, the dataset path in the config must point to a dataset file that exists on your machine.